# 📊 Notebook 01: Carregamento e Limpeza dos Dados
## Análise de Desigualdades de Gênero na Educação Básica Brasileira

**Autora:** Sara - Mestra em Educação

### 📋 Objetivos deste notebook:
1. Carregar os microdados do Censo Escolar 2025
2. Selecionar as variáveis relevantes para análise de gênero
3. Realizar limpeza e tratamento inicial dos dados
4. Salvar dados processados para análises subsequentes

---

In [1]:
# Importações necessárias
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Adicionar diretório de scripts ao path
import sys
sys.path.append('../scripts')

# Importar configurações
from config import RAW_DATA_PATH, PROCESSED_DATA_PATH, CORES

print("✅ Bibliotecas importadas com sucesso!")
print(f"📁 Caminho dos dados: {RAW_DATA_PATH}")

✅ Configurações carregadas!
📁 Dados brutos: /home/sara/Censo25/data/
📁 Dados processados: /home/sara/Censo25/data/processed/
📁 Gráficos: /home/sara/Censo25/outputs/figures/
✅ Bibliotecas importadas com sucesso!
📁 Caminho dos dados: /home/sara/Censo25/data/


## 1️⃣ Estrutura dos Dados do Censo 2025

Os dados são **agregados por escola**, não individuais.

### Dados de GÊNERO disponíveis:
- **Matrículas**: `QT_MAT_BAS_FEM`, `QT_MAT_BAS_MASC` (total da educação básica)
- **Gestores**: `QT_GEST_BAS_FEM`, `QT_GEST_BAS_MASC` (total de gestores)

---

In [2]:
# Definir colunas a serem carregadas

# Tabela Escola
cols_escola = [
    'CO_ENTIDADE', 'NO_ENTIDADE', 'CO_MUNICIPIO', 'NO_MUNICIPIO',
    'CO_UF', 'SG_UF', 'NO_REGIAO', 'CO_REGIAO',
    'TP_DEPENDENCIA', 'TP_LOCALIZACAO', 'TP_SITUACAO_FUNCIONAMENTO'
]

# Tabela Matrícula - APENAS colunas de gênero que existem!
cols_matricula = [
    'CO_ENTIDADE',
    'QT_MAT_BAS',        # Total matrículas básica
    'QT_MAT_BAS_FEM',    # Matrículas femininas
    'QT_MAT_BAS_MASC',   # Matrículas masculinas
    'QT_MAT_INF_CRE',    # Creche (total)
    'QT_MAT_INF_PRE',    # Pré-escola (total)
    'QT_MAT_FUND_AI',    # Fundamental Anos Iniciais (total)
    'QT_MAT_FUND_AF',    # Fundamental Anos Finais (total)
    'QT_MAT_MED',        # Ensino Médio (total)
    'QT_MAT_EJA'         # EJA (total)
]

# Tabela Gestor
cols_gestor = [
    'CO_ENTIDADE',
    'QT_GEST_BAS',       # Total gestores
    'QT_GEST_BAS_FEM',   # Gestores femininos
    'QT_GEST_BAS_MASC',  # Gestores masculinos
    'QT_GEST_BAS_DIRETOR',
    'QT_GEST_BAS_OUTRO'
]

print("📊 Colunas definidas com sucesso!")

📊 Colunas definidas com sucesso!


In [3]:
# Carregar Tabela Escola
print("🏫 Carregando Tabela Escola...")
escolas = pd.read_csv(
    RAW_DATA_PATH + 'Tabela_Escola_2025.csv',
    sep=';',
    encoding='latin1',
    usecols=cols_escola,
    low_memory=False
)

print(f"✅ {len(escolas):,} escolas carregadas")

🏫 Carregando Tabela Escola...


✅ 214,192 escolas carregadas


In [4]:
# Carregar Tabela Matrícula
print("📚 Carregando Tabela Matrícula...")
matriculas = pd.read_csv(
    RAW_DATA_PATH + 'Tabela_Matricula_2025.csv',
    sep=';',
    encoding='latin1',
    usecols=cols_matricula,
    low_memory=False
)

print(f"✅ {len(matriculas):,} registros de matrícula carregados")

📚 Carregando Tabela Matrícula...


✅ 178,766 registros de matrícula carregados


In [5]:
# Carregar Tabela Gestor
print("👨‍💼 Carregando Tabela Gestor...")
gestores = pd.read_csv(
    RAW_DATA_PATH + 'Tabela_Gestor_Escolar_2025.csv',
    sep=';',
    encoding='latin1',
    usecols=cols_gestor,
    low_memory=False
)

print(f"✅ {len(gestores):,} registros de gestores carregados")

👨‍💼 Carregando Tabela Gestor...
✅ 180,540 registros de gestores carregados


In [6]:
# VALIDAR códigos do INEP e filtrar escolas ativas
print("🔍 Validando códigos do INEP...")
print("\n📊 Valores únicos em TP_DEPENDENCIA:")
print(escolas['TP_DEPENDENCIA'].value_counts().sort_index())

print("\n📊 Valores únicos em TP_LOCALIZACAO:")
print(escolas['TP_LOCALIZACAO'].value_counts().sort_index())

print("\n📊 Valores únicos em TP_SITUACAO_FUNCIONAMENTO:")
print(escolas['TP_SITUACAO_FUNCIONAMENTO'].value_counts().sort_index())

# Filtrar escolas ativas
print("\n🧹 Filtrando escolas ativas (TP_SITUACAO_FUNCIONAMENTO == 1)...")
escolas_ativas = escolas[escolas['TP_SITUACAO_FUNCIONAMENTO'] == 1].copy()
print(f"✅ {len(escolas_ativas):,} escolas ativas")

# Mapear códigos para labels (confirmados com documentação INEP)
map_dependencia = {1: 'Federal', 2: 'Estadual', 3: 'Municipal', 4: 'Privada'}
escolas_ativas['TP_DEPENDENCIA'] = escolas_ativas['TP_DEPENDENCIA'].map(map_dependencia)

map_localizacao = {1: 'Urbana', 2: 'Rural'}
escolas_ativas['TP_LOCALIZACAO'] = escolas_ativas['TP_LOCALIZACAO'].map(map_localizacao)

print("\n✅ Labels criados com base na documentação do INEP!")

🔍 Validando códigos do INEP...

📊 Valores únicos em TP_DEPENDENCIA:
TP_DEPENDENCIA
1       728
2     33561
3    128145
4     51758
Name: count, dtype: int64

📊 Valores únicos em TP_LOCALIZACAO:
TP_LOCALIZACAO
1    144217
2     69975
Name: count, dtype: int64

📊 Valores únicos em TP_SITUACAO_FUNCIONAMENTO:
TP_SITUACAO_FUNCIONAMENTO
1    180540
2     29780
3      3872
Name: count, dtype: int64

🧹 Filtrando escolas ativas (TP_SITUACAO_FUNCIONAMENTO == 1)...
✅ 180,540 escolas ativas



✅ Labels criados com base na documentação do INEP!


In [7]:
# Mesclar tabelas
print("🔗 Mesclando tabelas...")

escolas_completas = pd.merge(escolas_ativas, matriculas, on='CO_ENTIDADE', how='left')
escolas_completas = pd.merge(escolas_completas, gestores, on='CO_ENTIDADE', how='left')

# Preencher valores nulos
cols_qt = [c for c in escolas_completas.columns if c.startswith('QT_')]
for col in cols_qt:
    escolas_completas[col] = escolas_completas[col].fillna(0)

# Calcular proporções
escolas_completas['PORC_MENINAS'] = (
    escolas_completas['QT_MAT_BAS_FEM'] / 
    (escolas_completas['QT_MAT_BAS_FEM'] + escolas_completas['QT_MAT_BAS_MASC']) * 100
).fillna(0)

escolas_completas['PORC_GESTORAS'] = (
    escolas_completas['QT_GEST_BAS_FEM'] / 
    (escolas_completas['QT_GEST_BAS_FEM'] + escolas_completas['QT_GEST_BAS_MASC']) * 100
).fillna(0)

print(f"✅ DataFrame criado com {len(escolas_completas):,} escolas")

🔗 Mesclando tabelas...


✅ DataFrame criado com 180,540 escolas


In [8]:
# Salvar dados processados
print("💾 Salvando dados processados...")

escolas_completas.to_csv(
    PROCESSED_DATA_PATH + 'escolas_analise.csv',
    index=False,
    encoding='utf-8'
)

matriculas.to_csv(
    PROCESSED_DATA_PATH + 'matriculas_agregadas.csv',
    index=False,
    encoding='utf-8'
)

gestores.to_csv(
    PROCESSED_DATA_PATH + 'gestores_agregados.csv',
    index=False,
    encoding='utf-8'
)

escolas_ativas.to_csv(
    PROCESSED_DATA_PATH + 'escolas_ativas.csv',
    index=False,
    encoding='utf-8'
)

print("✅ Dados salvos com sucesso!")

💾 Salvando dados processados...


✅ Dados salvos com sucesso!


## ✅ Resumo

### 📊 Dados Carregados:
- Escolas: ~190k registros
- Matrículas: ~190k registros
- Gestores: ~190k registros

### 📁 Arquivos Salvos:
- `escolas_analise.csv` - DataFrame principal
- `matriculas_agregadas.csv` - Matrículas por escola
- `gestores_agregados.csv` - Gestores por escola
- `escolas_ativas.csv` - Escolas em funcionamento

### 🔑 Colunas de Gênero:
- `PORC_MENINAS` - % de meninas por escola
- `PORC_GESTORAS` - % de gestoras por escola

### 🚀 Próximos Passos:
Ir para **02_analise_matriculas_por_genero.ipynb**

---

⚠️ **NOTA:** Dados de gênero estão disponíveis apenas para TOTAIS (não por etapa de ensino).

✨ **Notebook concluído!** ✨